In [1]:
import os
import sys
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Add the current directory to the path to import rejectlearn
sys.path.insert(0, os.path.abspath('.'))
from rejectlearn.rejection import RejectionClassifier


In [2]:
dataset_name = "diabetes" 
algorithm = "xgb"  # "tree", "rf", "xgb"
mechanism = "regva"  # "naive", "cv", "regva

In [4]:
# Define predictors and target
dataset = pd.read_csv(f"data/{dataset_name}.csv", sep=",")
X = dataset.drop(columns=["Class"]).values
y = pd.Series(dataset["Class"].values, dtype="float64")


if mechanism in ["regva", "avgva"]:
    results = {
        "rejection_rate": [],
        "estimate": [],
        "lower_bound": [],
        "upper_bound": [],
        "accuracy": [],
    }
else:
    results = {
        "rejection_rate": [],
        "estimate": [],
        "accuracy": [],
    }


for rejection_rate in tqdm([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]):

    results["rejection_rate"].append(rejection_rate)

    iter_accs = []
    iter_estimates = []
    iter_errors = []
    iter_lower_bounds = []
    iter_upper_bounds = []
    iter_covs = []

    for _ in tqdm(range(500)):

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y)


        # Split the training set into a proper training set and a calibration set
        if mechanism != "naive":
            X_prop, X_cal, y_prop, y_cal = train_test_split(X_train, y_train, test_size=0.33, stratify=y_train)

        # Train the underlying classifier
        if algorithm == "tree":
            if mechanism != "naive":
                clf = DecisionTreeClassifier(min_samples_leaf=round(np.sqrt(len(y_prop))))
            else:
                clf = DecisionTreeClassifier(min_samples_leaf=round(np.sqrt(len(y_train))))
        if algorithm == "rf":
            clf = RandomForestClassifier(n_jobs=-1)
        if algorithm == "xgb":
            clf = XGBClassifier(n_jobs=-1)
        
        if mechanism == "naive":
            rc = RejectionClassifier(estimator=clf, mechanism=mechanism, cv="prefit")
            rc.fit(X_train, y_train.values)
        else:
            clf.fit(X_prop, y_prop)
            rc = RejectionClassifier(estimator=clf, mechanism=mechanism, cv="prefit")
            rc.fit(X_cal, y_cal.values)
             
        metrics = rc.evaluate_selective_performance(
            X_test,
            y_test.values,
            rejection_rate=rejection_rate,
        )
        
        iter_accs.append(metrics["accuracy"])
        iter_estimates.append(metrics["estimate"])
        iter_errors.append(metrics["error"])
        
        if mechanism in ["regva", "avgva"]:
            iter_lower_bounds.append(metrics["lower_bound"])
            iter_upper_bounds.append(metrics["upper_bound"])

    results["estimate"].append(np.mean(iter_estimates))
    results["accuracy"].append(np.mean(iter_accs))

    if mechanism in ["regva", "avgva"]:
        results["lower_bound"].append(np.mean(iter_lower_bounds))
        results["upper_bound"].append(np.mean(iter_upper_bounds))

    #pd.DataFrame(results).to_csv(f"results/{dataset_name}_{mechanism}_{algorithm}.csv", index=False)

results = pd.DataFrame(results)


100%|██████████| 9/9 [00:30<00:00,  3.36s/it]


In [5]:
results

,rejection_rate,estimate,lower_bound,upper_bound,accuracy
0,0.1,0.742321,0.727479,0.786933,0.745903
1,0.2,0.755758,0.741437,0.801197,0.764617
2,0.3,0.778466,0.764768,0.826871,0.786873
3,0.4,0.796850,0.784456,0.846573,0.812381
4,0.5,0.828010,0.816909,0.880693,0.848792
5,0.6,0.853112,0.843458,0.905899,0.860231
6,0.7,0.880074,0.872510,0.933070,0.892173
7,0.8,0.904355,0.898214,0.957890,0.929310
8,0.9,0.926539,0.922104,0.978702,0.954463
